In [ ]:
require(Seurat)
require(scales)
require(ggplot2)
require(viridis)
require(dplyr)
require(GeneTrajectory)
require(Matrix)
require(plot3D)

In [ ]:
reticulate::use_python("/home/xili/anaconda3/envs/scRepertoire_env/bin/python", required = TRUE)
library(sceasy)
library(reticulate)
py_config()

In [ ]:
#set the directory for outputs
project_ID="pool_T"
mtx="transcriptome"

In [ ]:
outputdir=paste0("transcriptome/",project_ID)

In [ ]:
sceasy::convertFormat(paste0(outputdir,"_figures/",project_ID,"_anno_scvi.h5ad"),from="anndata", to="seurat", 
                      outFile = paste0(outputdir,"_DEG/",project_ID,"_anno_scvi.rds"))

In [ ]:
data <- readRDS(paste0(outputdir,"_DEG/",project_ID,"_anno_raw.rds"))

In [ ]:
treatment_g="all"
data_S <- data

In [ ]:
DimPlot(data_S, group.by = "anno_sub", shuffle = T)

In [ ]:
DimPlot(data_S, group.by = "treatment", shuffle = T)

In [ ]:
assay <- "RNA"
DefaultAssay(data_S) <- assay
data_S <- NormalizeData(data_S)
data_S <- FindVariableFeatures(data_S, nfeatures = 6000)

data_S <- ScaleData(data_S)
data_S <- RunPCA(data_S)
all_genes <- data_S@assays[[assay]]@var.features
expr_percent <- apply(as.matrix(data_S[[assay]]@data[all_genes, ]) > 0, 1, sum)/ncol(data_S)
genes <- all_genes[which(expr_percent > 0.05 & expr_percent < 0.6)]
length(genes)

## Prepare the input for gene-gene distance computation

Next, we compute the Diffusion Map cell embedding using RunDM. We then use GetGraphDistance to construct the cell-cell kNN graph (K=10 in this example) and calculate cell-cell graph distances (defined by the length of the shortest path connecting two cells in the kNN graph). To improve computation efficiency, we coarse-grain the cell graph by grouping cells into N=500 “meta-cells”.

In [ ]:
# Compute the Diffusion Map cell embedding
data_S <- GeneTrajectory::RunDM(data_S)
# Calculate cell-cell graph distances over a cell-cell kNN graph
cell.graph.dist <- GetGraphDistance(data_S, K = 10)
# Coarse-grain the cell graph by grouping cells into `N`=500 "meta-cells"
cg_output <- CoarseGrain(data_S, cell.graph.dist, genes, N = 500)

## Compute gene-gene distances

The computation of gene-gene Wasserstein distances is by employing the package of Python Optimal Transport (POT). Here, we recommend users to set up a virtualenv using reticulate R package.

In [ ]:
# Import the function to compute gene-gene distances
cal_ot_mat_from_numpy <- reticulate::import('gene_trajectory.compute_gene_distance_cmd')$cal_ot_mat_from_numpy
# Compute gene-gene distances 
gene.dist.mat <- cal_ot_mat_from_numpy(ot_cost = cg_output[["graph.dist"]], gene_expr = cg_output[["gene.expression"]], num_iter_max = 5000, 
                                       show_progress_bar = TRUE)
rownames(gene.dist.mat) <- cg_output[["features"]]
colnames(gene.dist.mat) <- cg_output[["features"]]
dim(gene.dist.mat)

In [ ]:
K=4
gene_embedding <- GetGeneEmbedding(gene.dist.mat, K = K)$diffu.emb

In [ ]:
# Extract 3 gene trajectories
gene_trajectory <- ExtractGeneTrajectory(gene_embedding, gene.dist.mat, N = 4, t.list = c(7,5,3,3), K = K)
table(gene_trajectory$selected)

In [ ]:
# Visualize gene trajectories
pdf(paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_embed_3D.pdf"),width=3, height =3, pointsize = 8)
par(mar = c(1.5,1.5,1.5,1.5))
x=1
y=2
z=5
selected_genes <- c("Cd163","Mrc1","H2-Oa","Ifitm6","H2-DMb2",
                   "H2-Ob","Ifi205","Ifi27l2a","Arg1","Cd80","Cd86","Cd40","Cxcl10","Mafb")

# Find indices of selected genes
gene_indices <- which(rownames(gene_embedding) %in% selected_genes)

scatter3D(gene_embedding[,x],
          gene_embedding[,y],
          gene_embedding[,z],
          bty = "b2", 
          colvar = (as.integer(as.factor(gene_trajectory$selected))),
          main = "trajectory", 
          pch = 19, 
          cex = 0.2, 
          alpha=0.7,
          theta = 90, 
          phi = 22,
          col = ramp.col(c("orange", "purple", "brown","darkgreen")),
          xlim = c(-0.02, -0.005),
          ylim = c(-0.01, 0.03),
          zlim = c(-0.02, 0.007),
          #clim = range((as.integer(as.factor(gene_trajectory$selected)))),
          ticktype = "detailed")


dev.off()

In [ ]:
# Visualize gene trajectories
pdf(paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_embed_3D_text2.pdf"),width=3, height =3, pointsize = 8)
par(mar = c(1.5,1.5,1.5,1.5))
x=1
y=2
z=5
selected_genes <- c("C3ar1","C5ar1","Cd209a","Plac8","Cd40","Cd163","Lck","Il12","Il18")

# Find indices of selected genes
gene_indices <- which(rownames(gene_embedding) %in% selected_genes)

scatter3D(gene_embedding[gene_indices,x],
          gene_embedding[gene_indices,y],
          gene_embedding[gene_indices,z],
          bty = "b2", 
          colvar = (as.integer(as.factor(gene_trajectory[gene_indices,]$selected))),
          main = "trajectory", 
          pch = 19, 
          cex = 0.2, 
          theta = 90, 
          phi = 22,
          col = ramp.col(c("orange", "purple", "brown","darkgreen")),
          xlim = c(-0.02, -0.005),
          ylim = c(-0.01, 0.03),
          zlim = c(-0.02, 0.007),
          #clim = range((as.integer(as.factor(gene_trajectory$selected)))),
          ticktype = "detailed")


# Add connecting lines first
# Define multiple offset directions to spread out labels
offset_directions <- list(
  c(0.002, 0.002, 0.002),   # Top-right
  c(0.002, -0.002, 0.002),  # Bottom-right
  c(-0.002, 0.002, 0.002),  # Top-left
  c(-0.002, -0.002, 0.002)  # Bottom-left
)

# Apply different offsets to each gene
for(i in seq_along(gene_indices)) {
  offset <- offset_directions[[(i-1) %% length(offset_directions) + 1]]
  
  segments3D(
    x0 = gene_embedding[gene_indices[i], x],
    y0 = gene_embedding[gene_indices[i], y],
    z0 = gene_embedding[gene_indices[i], z],
    x1 = gene_embedding[gene_indices[i], x] + offset[1],
    y1 = gene_embedding[gene_indices[i], y] + offset[2],
    z1 = gene_embedding[gene_indices[i], z] + offset[3],
    add = TRUE,
    col = "gray",
    lwd = 1
  )
  
  text3D(gene_embedding[gene_indices[i], x] + offset[1],
         gene_embedding[gene_indices[i], y] + offset[2],
         gene_embedding[gene_indices[i], z] + offset[3],
         labels = rownames(gene_embedding)[gene_indices[i]],
         add = TRUE,
         cex = 0.7,
         col = "black",
         font = 2)
}
dev.off()

In [ ]:
# Visualize gene trajectories
pdf(paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_embed_3D_text_tmp.pdf"),width=3, height =3, pointsize = 8)
par(mar = c(1.5,1.5,1.5,1.5))
x=1
y=2
z=5
selected_genes <- c("C3ar1","C5ar1","Cd209a","Plac8",)

# Find indices of selected genes
gene_indices <- which(rownames(gene_embedding) %in% selected_genes)

scatter3D(gene_embedding[,x],
          gene_embedding[,y],
          gene_embedding[,z],
          bty = "b2", 
          colvar = (as.integer(as.factor(gene_trajectory[,]$selected))),
          main = "trajectory", 
          pch = 19, 
          cex = 0.2, 
          theta = 90, 
          phi = 22,
          col = ramp.col(c("orange", "purple", "brown","darkgreen")),
          xlim = c(-0.02, -0.005),
          ylim = c(-0.01, 0.03),
          zlim = c(-0.02, 0.007),
          #clim = range((as.integer(as.factor(gene_trajectory$selected)))),
          ticktype = "detailed")


# Add connecting lines first
# Define multiple offset directions to spread out labels
offset_directions <- list(
  c(0.002, 0.002, 0.002),   # Top-right
  c(0.002, -0.002, 0.002),  # Bottom-right
  c(-0.002, 0.002, 0.002),  # Top-left
  c(-0.002, -0.002, 0.002)  # Bottom-left
)

# Apply different offsets to each gene
for(i in seq_along(gene_indices)) {
  offset <- offset_directions[[(i-1) %% length(offset_directions) + 1]]
  
  segments3D(
    x0 = gene_embedding[gene_indices[i], x],
    y0 = gene_embedding[gene_indices[i], y],
    z0 = gene_embedding[gene_indices[i], z],
    x1 = gene_embedding[gene_indices[i], x] + offset[1],
    y1 = gene_embedding[gene_indices[i], y] + offset[2],
    z1 = gene_embedding[gene_indices[i], z] + offset[3],
    add = TRUE,
    col = "gray",
    lwd = 1
  )
  
  text3D(gene_embedding[gene_indices[i], x] + offset[1],
         gene_embedding[gene_indices[i], y] + offset[2],
         gene_embedding[gene_indices[i], z] + offset[3],
         labels = rownames(gene_embedding)[gene_indices[i]],
         add = TRUE,
         cex = 0.7,
         col = "black",
         font = 2)
}
dev.off()

In [ ]:
# Visualize gene trajectories
pdf(paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_embed_2D.pdf"),width=3, height =3, pointsize = 8)
par(mar = c(1.5,1.5,1.5,1.5))
x=1
y=2
z=5
scatter2D(gene_embedding[,x],
          gene_embedding[,y],
          #gene_embedding[,z],
          bty = "b2", colvar = (as.integer(as.factor(gene_trajectory$selected))),
         
          col = ramp.col(c("orange", "purple", "brown","darkgreen")) ,
         # Control Z-axis scaling
          
          clim = range((as.integer(as.factor(gene_trajectory$selected)))),  # Color limits
          ticktype = "detailed")
dev.off()

In [ ]:
###### gene_list <- list()
for (i in 1:){
  gene_trajectory_sub <- gene_trajectory[which(gene_trajectory$selected == paste0("Trajectory-", i)),]
  genes <- rownames(gene_trajectory_sub)[order(gene_trajectory_sub[, paste0("Pseudoorder", i)])]
  gene_list[[i]] <- genes
}

In [ ]:
library(SeuratWrappers)

In [ ]:
data_S <- RunALRA(data_S)

In [ ]:
data_S <- AddGeneBinScore(data_S, gene_trajectory, N.bin = 6, trajectories = 1:4, assay = "alra", reverse = c(F, F, T))

In [ ]:
pdf(paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_1.pdf"),width=10, height =6, pointsize = 4)
# Visualize gene bin plots for each gene trajectory
FeaturePlot(data_S, pt.size = 0.05, features = paste0("Trajectory",1,"_genes", 1:6), ncol = 3, order = T) &
  scale_color_gradientn(colors = rev(brewer_pal(palette = "RdYlBu")(10))) & NoLegend() & NoAxes() & theme(title = element_text(size = 10))
dev.off()

In [ ]:
pdf(paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_2.pdf"),width=10, height =6, pointsize = 4)
FeaturePlot(data_S, pt.size = 0.05, features = paste0("Trajectory",2,"_genes", 1:6), ncol = 3, order = T) &
  scale_color_gradientn(colors = rev(brewer_pal(palette = "RdYlBu")(10))) & NoLegend() & NoAxes() & theme(title = element_text(size = 10))
dev.off()

In [ ]:
pdf(paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_3.pdf"),width=10, height =6, pointsize = 4)
FeaturePlot(data_S, pt.size = 0.01, features = paste0("Trajectory",3,"_genes", 1:6), ncol = 3, order = T) &
  scale_color_gradientn(colors = rev(brewer_pal(palette = "RdYlBu")(10))) & NoLegend() & NoAxes() & theme(title = element_text(size = 10))
dev.off()

In [ ]:
pdf(paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_4.pdf"),width=10, height =6, pointsize = 4)
FeaturePlot(data_S, pt.size = 0.01, features = paste0("Trajectory",4,"_genes", 1:6), ncol = 3, order = T) &
  scale_color_gradientn(colors = rev(brewer_pal(palette = "RdYlBu")(10))) & NoLegend() & NoAxes() & theme(title = element_text(size = 10))
dev.off()

In [ ]:
Traj_3 = gene_trajectory[gene_trajectory$selected=="Trajectory-3",]
write.csv(Traj_3[order(Traj_3$Pseudoorder3),][1:2000,],paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_3.csv"))

In [ ]:
Traj_2 = gene_trajectory[gene_trajectory$selected=="Trajectory-2",]
write.csv(Traj_2[order(Traj_2$Pseudoorder2),][1:2000,],paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_2.csv"))

In [ ]:
Traj_1 = gene_trajectory[gene_trajectory$selected=="Trajectory-1",]
write.csv(Traj_1[order(Traj_1$Pseudoorder1),][1:2000,],paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_1.csv"))

In [ ]:
Traj_4 = gene_trajectory[gene_trajectory$selected=="Trajectory-4",]
write.csv(gene_trajectory,paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory.csv"))

In [ ]:
write.csv(Traj_4[order(Traj_4$Pseudoorder4),][1:2000,],paste0(outputdir,project_ID,"_",treatment_g,"_","_Trajectory_4.csv"))

In [ ]:
data_S@meta.data$treatment_anno=paste0(data_S@meta.data$anno_sub,"_",data_S@meta.data$treatment)

In [ ]:
p<-DotPlot(data_S, 
        features = c('Trajectory4_genes1','Trajectory4_genes2','Trajectory4_genes3',
                     'Trajectory4_genes4','Trajectory4_genes5','Trajectory4_genes6'),
        cols = c("lightgrey", "red"),  # Color scale
        dot.scale = 6,                 # Dot size
        col.min = -1,                # Minimum expression level
        col.max = 1,
       group.by = "treatment") +               # Maximum expression level
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        axis.title = element_blank()) +
  labs(title = "Customized Dot Plot") +
  coord_flip()  # Flip coordinates if preferred

options(repr.plot.width = 12, repr.plot.height = 8)  # For Jupyter/RMarkdown
print(p)

In [ ]:
DotPlot(data_S, 
        features = c('Trajectory3_genes1','Trajectory3_genes2','Trajectory3_genes3','Trajectory3_genes4','Trajectory3_genes5','Trajectory3_genes6'),
        cols = c("lightgrey", "red"),  # Color scale
        dot.scale = 6,                 # Dot size
        col.min = -1,                # Minimum expression level
        col.max = 1,
       group.by = "treatment",cluster.idents = TRUE) +               # Maximum expression level
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        axis.title = element_blank()) +
  labs(title = "Customized Dot Plot") +
  coord_flip()  # Flip coordinates if preferred